In [1]:
python src\exp20_dreams_vs_direct.py

SyntaxError: unexpected character after line continuation character (501366138.py, line 1)

In [2]:
python xp20_dreams_vs_direct.py

SyntaxError: invalid syntax (3262476052.py, line 1)

In [3]:
python exp20_dreams_vs_direct.py

SyntaxError: invalid syntax (3554104904.py, line 1)

In [4]:
!python src/exp20_dreams_vs_direct.py

C:\Users\Bo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe: can't open file 'C:\\Users\\Bo\\projects\\neuromorphic-sandbox\\src\\src\\exp20_dreams_vs_direct.py': [Errno 2] No such file or directory


In [7]:
import os
os.chdir(r"C:\Users\Bo\projects\neuromorphic-sandbox")
import sys
sys.path.insert(0, "src")

In [8]:
import numpy as np, time
from sparse_state import SparsePairState
from spatial import SpatialGrid

PATTERNS = [(0,3),(1,4),(2,)]
N, NC, DREAM_END, REV, TOTAL = 1000, 5, 150, 600, 1400

def make_adv():
    centers = np.array([[10,10],[30,10],[20,34],[30,16],[10,16]], dtype=float)
    rng = np.random.default_rng(1)
    pts, cids = [], []
    for cid,(cx,cy) in enumerate(centers):
        for _ in range(200):
            th,r = rng.uniform(0,2*np.pi), rng.uniform(0,5)
            pts.append([cx+r*np.cos(th), cy+r*np.sin(th)])
            cids.append(cid)
    return np.array(pts), np.array(cids)

def run_arm(coords, cids, seed, dream):
    rng3 = np.random.default_rng(seed)
    rng2 = np.random.default_rng(7)
    src = rng2.integers(0,N,10000); dst = rng2.integers(0,N,10000)
    keep = src!=dst; src,dst = src[keep],dst[keep]
    inhib = rng2.random(N)<0.20
    v = np.zeros(N); refrac = np.zeros(N,dtype=int)
    C = SparsePairState(0.95); E = SparsePairState(0.90); V = SparsePairState(0.999)
    Rhat = np.zeros(3)
    g = SpatialGrid(coords, 25.0)
    nbr = [g.within(i,25.0) for i in range(N)]
    D2 = ((coords[:,None,:]-coords[None,:,:])**2).sum(-1)
    out_t = [[] for _ in range(N)]
    out_w = [[] for _ in range(N)]
    for s2,d2 in zip(src,dst):
        out_t[s2].append(d2); out_w[s2].append(-0.60 if inhib[s2] else 0.30)
    swap = 1000 if dream else 500
    for t in range(TOTAL):
        p = (t//20)%3
        if dream and t==DREAM_END: swap=500
        if dream and t<DREAM_END:
            inp = rng3.uniform(0,0.02,N)
            if (t%20)<5:
                for c in PATTERNS[p]: inp[cids==c]+=0.35
        elif t<REV:
            inp = rng3.uniform(0,0.02,N)
            if (t%20)<5:
                for c in PATTERNS[p]: inp[cids==c]+=0.5
        else:
            inp = rng3.uniform(0,0.02,N)
            rp = [(1,4),(0,3),(2,)]; pat=rp[p]
            if (t%20)<5:
                for c in pat: inp[cids==c]+=0.5
        v_ = v*0.90+inp; fired=(v_>=1.0)&(refrac==0); f=np.where(fired)[0]
        if len(f):
            for fi in f:
                for ti,wi in zip(out_t[fi],out_w[fi]): v_[ti]+=wi
        C.tick(); E.tick(); V.tick()
        if len(f):
            fs=set(int(x) for x in f)
            for i in f:
                i=int(i)
                for j in nbr[i]:
                    if int(j) in fs: C.deposit(i,int(j),1.0); E.deposit(i,int(j),1.0)
        v=np.maximum(v_,0); v[fired]=0; refrac[fired]=3; refrac[refrac>0]-=1
        if (t%20)==6:
            base = {0:1.0,1:0.0,2:-1.0}[p] if t<REV else {0:0.0,1:1.0,2:-1.0}[p]
            delta=base-Rhat[p]
            if abs(delta)>1e-9:
                E.prune_below(1e-6)
                for key in list(E.store.keys()):
                    ev=E.get(*key)
                    if ev!=0: V.deposit(key[0],key[1],delta*ev)
            Rhat[p]+=0.15*delta
        if (t+1)%40==0:
            C.prune_below(1e-6)
            sc=np.array([V.get(int(src[k]),int(dst[k])) for k in range(len(src))])
            cold=np.argsort(sc)[:swap]
            ci,cj,_=C.get_arrays()
            if len(ci)==0: continue
            ci,cj=ci[ci!=cj],cj[ci!=cj]  # drop self
            ex=set(zip(src.tolist(),dst.tolist()))
            mk=np.array([(int(a),int(b)) not in ex for a,b in zip(ci,cj)])
            ci,cj=ci[mk],cj[mk]
            if len(ci)==0: continue
            dd=D2[ci,cj]
            vp=np.maximum(np.array([V.get(int(a),int(b)) for a,b in zip(ci,cj)]),0)
            cp=np.array([C.get(int(a),int(b)) for a,b in zip(ci,cj)])
            score=(vp+0.01*cp)/(1+0.05*dd); pos=score>0
            ci,cj,score=ci[pos],cj[pos],score[pos]
            if len(ci)==0: continue
            order=np.argsort(score)[::-1][:len(cold)]
            n2=min(len(cold),len(order))
            src[cold[:n2]]=ci[order[:n2]]; dst[cold[:n2]]=cj[order[:n2]]
            out_t=[[] for _ in range(N)]; out_w=[[] for _ in range(N)]
            for s2,d2b in zip(src,dst):
                out_t[s2].append(d2b); out_w[s2].append(-0.60 if inhib[s2] else 0.30)
    M=np.zeros((NC,NC),dtype=int); np.add.at(M,(cids[src],cids[dst]),1)
    return M[0,3]+M[3,0], M[1,4]+M[4,1], M.sum()-np.trace(M)-(M[0,3]+M[3,0])-(M[1,4]+M[4,1]), D2[src,dst].sum()

print("="*70)
print("EXP 20 (Grok): FACTORY DREAMS vs DIRECT RPE — adversarial body")
print("="*70)
RA,RB=[],[]
for s in [0,1,2,3,4]:
    t0=time.time(); co,ci=make_adv()
    a=run_arm(co.copy(),ci.copy(),s,True); b=run_arm(co.copy(),ci.copy(),s,False)
    RA.append(a); RB.append(b)
    print("  seed {}: A=old:{} new:{} sup:{} e:{:,.0f} | B=old:{} new:{} sup:{} e:{:,.0f}  ({:.0f}s)".format(
        s,a[0],a[1],a[2],a[3],b[0],b[1],b[2],b[3],time.time()-t0))
A=np.array(RA,dtype=float); B=np.array(RB,dtype=float)
labs=["residual old","new acquired","superstition","wire energy"]
print("\n"+"="*70)
for i,l in enumerate(labs):
    am,ast,bm,bst=A[:,i].mean(),A[:,i].std(),B[:,i].mean(),B[:,i].std()
    pct=(am-bm)/max(abs(bm),1)*100
    print("  {:>15}: A={:>7,.0f}+/-{:>5,.0f}  B={:>7,.0f}+/-{:>5,.0f}  delta={:>+6.0f}%".format(l,am,ast,bm,bst,pct))

EXP 20 (Grok): FACTORY DREAMS vs DIRECT RPE — adversarial body
  seed 0: A=old:0 new:0 sup:62 e:69,599 | B=old:0 new:0 sup:0 e:64,728  (720s)
  seed 1: A=old:0 new:0 sup:63 e:79,074 | B=old:0 new:0 sup:2 e:58,472  (4015s)


KeyboardInterrupt: 